In [1]:
import pandas as pd
import os

# ۱. تعریف لیست نام فایل‌ها
file_names = [
    'DEMO_J.xpt', 'BMX_J.xpt', 'BPX_J.xpt', 'DR1TOT_J.xpt', 
    'DR2TOT_J.xpt', 'DXX_J.xpt', 'HDL_J.xpt', 'PAQ_J.xpt', 
    'SLQ_J.xpt', 'TRIGLY_J.xpt'
]

# ۲. تعریف مسیر پوشه فایل‌های خام XPT
raw_data_dir = os.path.join("..", "data", "raw", "nhanes")

# ۳. خواندن فایل دموگرافیک به عنوان فایل پایه
base_file_path = os.path.join(raw_data_dir, file_names[0])
merged_df = pd.read_sas(base_file_path)

# ۴. ادغام زنجیره‌ای بقیه فایل‌ها بر اساس ستون SEQN
for file in file_names[1:]:
    print(f"در حال ادغام فایل: {file}...")
    next_file_path = os.path.join(raw_data_dir, file)
    next_df = pd.read_sas(next_file_path)
    
    # استفاده از how='left' برای حفظ تمام افراد فایل دموگرافیک
    merged_df = pd.merge(merged_df, next_df, on='SEQN', how='left')

# ۵. ذخیره فایل یکپارچه شده در پوشه interim
output_interim_path = os.path.join("..", "data", "interim", "NHANES_Merged_Data.csv")
merged_df.to_csv(output_interim_path, index=False)
print(f"✅ ادغام با موفقیت انجام شد! فایل در مسیر زیر ذخیره شد:\n{output_interim_path}")

در حال ادغام فایل: BMX_J.xpt...
در حال ادغام فایل: BPX_J.xpt...
در حال ادغام فایل: DR1TOT_J.xpt...
در حال ادغام فایل: DR2TOT_J.xpt...
در حال ادغام فایل: DXX_J.xpt...
در حال ادغام فایل: HDL_J.xpt...
در حال ادغام فایل: PAQ_J.xpt...
در حال ادغام فایل: SLQ_J.xpt...
در حال ادغام فایل: TRIGLY_J.xpt...
✅ ادغام با موفقیت انجام شد! فایل در مسیر زیر ذخیره شد:
..\data\interim\NHANES_Merged_Data.csv


In [2]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import MinMaxScaler

print("در حال بارگذاری داده‌های اصلی از پوشه interim...")
input_path = os.path.join("..", "data", "interim", "NHANES_Merged_Data.csv")
df = pd.read_csv(input_path)

# ۱. فیلتر افراد بالای ۱۸ سال
df_clean = df[df['RIDAGEYR'] >= 18].copy()

# ۲. پر کردن داده‌های مفقود بیوشیمیایی و آزمایشگاهی با میانه (Imputation)
df_clean['DXDTOLE'] = df_clean['DXDTOLE'].fillna(df_clean['DXDTOLE'].median())
df_clean['LBDHDD'] = df_clean['LBDHDD'].fillna(df_clean['LBDHDD'].median())
df_clean['LBXTR'] = df_clean['LBXTR'].fillna(df_clean['LBXTR'].median())
df_clean['BMXWT'] = df_clean['BMXWT'].fillna(df_clean['BMXWT'].median())
df_clean['BMXHT'] = df_clean['BMXHT'].fillna(df_clean['BMXHT'].median())
df_clean['BMXBMI'] = df_clean['BMXBMI'].fillna(df_clean['BMXBMI'].median())

# ۳. ساخت شاخص ژنتیکی/متابولیکی (Genetic_Score) بین ۰ تا ۱۰۰
scaler = MinMaxScaler(feature_range=(0, 100))
df_clean['Raw_Genetic_Score'] = (df_clean['DXDTOLE'] / df_clean['DXDTOLE'].median()) + \
                                (df_clean['LBDHDD'] / df_clean['LBDHDD'].median()) - \
                                (df_clean['LBXTR'] / df_clean['LBXTR'].median())

df_clean['Genetic_Score'] = scaler.fit_transform(df_clean[['Raw_Genetic_Score']])
df_clean['Genetic_Score'] = df_clean['Genetic_Score'].round(2)

# ۴. مهندسی شاخص فعالیت بدنی (Activity_Score)
df_clean['Vigorous_Activity'] = df_clean['PAQ650'].apply(lambda x: 1 if x == 1 else 0)
df_clean['Moderate_Activity'] = df_clean['PAQ665'].apply(lambda x: 1 if x == 1 else 0)
df_clean['Activity_Score'] = df_clean['Vigorous_Activity'] * 2 + df_clean['Moderate_Activity']

# ۵. محاسبه پروتئین مورد نیاز به گرم (Protein_Requirement_g)
df_clean['Protein_Requirement_g'] = df_clean['BMXWT'] * (0.8 + (df_clean['Activity_Score'] * 0.2) + (df_clean['Genetic_Score'] / 500))
df_clean['Protein_Requirement_g'] = df_clean['Protein_Requirement_g'].round(2)

# ۶. مرتب‌سازی و انتخاب ستون‌های نهایی استاندارد
final_cols = {
    'SEQN': 'ID',
    'RIDAGEYR': 'Age',
    'RIAGENDR': 'Gender',
    'BMXWT': 'Weight_kg',
    'BMXHT': 'Height_cm',
    'BMXBMI': 'BMI',
    'Genetic_Score': 'Genetic_Score',
    'Activity_Score': 'Activity_Score',
    'Protein_Requirement_g': 'Protein_Requirement_g'
}

df_final = df_clean[list(final_cols.keys())].rename(columns=final_cols)
df_final['Gender'] = df_final['Gender'].map({1: 'Male', 2: 'Female'})

# ۷. ذخیره و خروجی گرفتن نهایی در پوشه processed
output_file = os.path.join("..", "data", "processed", "NHANES_Final_Dataset_with_Genetic_Score.csv")
df_final.to_csv(output_file, index=False)

print(f"✅ عملیات با موفقیت انجام شد! تعداد رکوردهای نهایی: {df_final.shape[0]}")
print(f"📁 فایل شما در مسیر پردازش‌شده‌ها ذخیره شد: {output_file}")

در حال بارگذاری داده‌های اصلی از پوشه interim...
✅ عملیات با موفقیت انجام شد! تعداد رکوردهای نهایی: 5856
📁 فایل شما در مسیر پردازش‌شده‌ها ذخیره شد: ..\data\processed\NHANES_Final_Dataset_with_Genetic_Score.csv


In [3]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import MinMaxScaler

print("در حال بارگذاری داده‌های اصلی از پوشه interim...")
input_path = os.path.join("..", "data", "interim", "NHANES_Merged_Data.csv")
df = pd.read_csv(input_path)

# ۱. فیلتر افراد بالای ۱۸ سال
df_clean = df[df['RIDAGEYR'] >= 18].copy()

# ۲. پر کردن داده‌های مفقود بیوشیمیایی و آزمایشگاهی با میانه
df_clean['DXDTOLE'] = df_clean['DXDTOLE'].fillna(df_clean['DXDTOLE'].median())
df_clean['LBDHDD'] = df_clean['LBDHDD'].fillna(df_clean['LBDHDD'].median())
df_clean['LBXTR'] = df_clean['LBXTR'].fillna(df_clean['LBXTR'].median())
df_clean['BMXWT'] = df_clean['BMXWT'].fillna(df_clean['BMXWT'].median())
df_clean['BMXHT'] = df_clean['BMXHT'].fillna(df_clean['BMXHT'].median())
df_clean['BMXBMI'] = df_clean['BMXBMI'].fillna(df_clean['BMXBMI'].median())

# ۳. ساخت شاخص ژنتیکی/متابولیکی (Genetic_Score) بین ۰ تا ۱۰۰
scaler = MinMaxScaler(feature_range=(0, 100))
df_clean['Raw_Genetic_Score'] = (df_clean['DXDTOLE'] / df_clean['DXDTOLE'].median()) + \
                                (df_clean['LBDHDD'] / df_clean['LBDHDD'].median()) - \
                                (df_clean['LBXTR'] / df_clean['LBXTR'].median())

df_clean['Genetic_Score'] = scaler.fit_transform(df_clean[['Raw_Genetic_Score']])
df_clean['Genetic_Score'] = df_clean['Genetic_Score'].round(2)

# ۴. مهندسی شاخص فعالیت بدنی (Activity_Score)
df_clean['Vigorous_Activity'] = df_clean['PAQ650'].apply(lambda x: 1 if x == 1 else 0)
df_clean['Moderate_Activity'] = df_clean['PAQ665'].apply(lambda x: 1 if x == 1 else 0)
df_clean['Activity_Score'] = df_clean['Vigorous_Activity'] * 2 + df_clean['Moderate_Activity']

# ۵. محاسبه پروتئین مورد نیاز به گرم با استفاده از ستون‌های اصلی NHANES
df_clean['Protein_Requirement_g'] = df_clean['BMXWT'] * (0.8 + (df_clean['Activity_Score'] * 0.2) + (df_clean['Genetic_Score'] / 500))
df_clean['Protein_Requirement_g'] = df_clean['Protein_Requirement_g'].round(2)

# ۶. مرتب‌سازی و تغییر نام ستون‌های نهایی به فرمت استاندارد و تمیز
final_cols = {
    'SEQN': 'ID',
    'RIDAGEYR': 'Age',
    'RIAGENDR': 'Gender',
    'BMXWT': 'Weight_kg',
    'BMXHT': 'Height_cm',
    'BMXBMI': 'BMI',
    'Genetic_Score': 'Genetic_Score',
    'Activity_Score': 'Activity_Score',
    'Protein_Requirement_g': 'Protein_Requirement_g'
}

df_final = df_clean[list(final_cols.keys())].rename(columns=final_cols)
df_final['Gender'] = df_final['Gender'].map({1: 'Male', 2: 'Female'})

# ۷. ذخیره و خروجی گرفتن نهایی در پوشه processed
output_file = os.path.join("..", "data", "processed", "NHANES_Final_Dataset_with_Genetic_Score.csv")
df_final.to_csv(output_file, index=False)

print("-" * 50)
print(f"✅ عملیات با موفقیت انجام شد!")
print(f"📊 تعداد رکوردهای نهایی بزرگسالان: {df_final.shape[0]}")
print(f"📁 فایل شما در مسیر زیر ذخیره شد:\n{output_file}")
print("-" * 50)

در حال بارگذاری داده‌های اصلی از پوشه interim...
--------------------------------------------------
✅ عملیات با موفقیت انجام شد!
📊 تعداد رکوردهای نهایی بزرگسالان: 5856
📁 فایل شما در مسیر زیر ذخیره شد:
..\data\processed\NHANES_Final_Dataset_with_Genetic_Score.csv
--------------------------------------------------


In [4]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import MinMaxScaler

print("در حال ساخت دیتاست مستر با استانداردهای Senior ML Engineer...")

# بارگذاری فایل اصلی ادغام شده از پوشه interim
input_path = os.path.join("..", "data", "interim", "NHANES_Merged_Data.csv")
df = pd.read_csv(input_path)

# ۱. فیلتر افراد بزرگسال (بالای ۱۸ سال)
df_clean = df[df['RIDAGEYR'] >= 18].copy()

# ۲. پاک‌سازی داده‌ها و پر کردن مقادیر خالی با میانه (Imputation)
df_clean['DXDTOLE'] = df_clean['DXDTOLE'].fillna(df_clean['DXDTOLE'].median())
df_clean['DXDTOPF'] = df_clean['DXDTOPF'].fillna(df_clean['DXDTOPF'].median())
df_clean['LBDHDD'] = df_clean['LBDHDD'].fillna(df_clean['LBDHDD'].median())
df_clean['LBXTR'] = df_clean['LBXTR'].fillna(df_clean['LBXTR'].median())
df_clean['BMXWT'] = df_clean['BMXWT'].fillna(df_clean['BMXWT'].median())
df_clean['BMXHT'] = df_clean['BMXHT'].fillna(df_clean['BMXHT'].median())
df_clean['BMXBMI'] = df_clean['BMXBMI'].fillna(df_clean['BMXBMI'].median())

# ۳. مهندسی ویژگی‌های ساختاری بدن (Body Composition)
df_clean['Lean_Mass_kg'] = (df_clean['DXDTOLE'] / 1000).round(2) # تبدیل گرم به کیلوگرم
df_clean['Body_Fat_Percent'] = df_clean['DXDTOPF'].round(2)

# ۴. استخراج میزان مصرف پروتئین واقعی فعلی فرد
df_clean['Daily_Protein_Intake_g'] = df_clean[['DR1TPROT', 'DR2TPROT']].mean(axis=1)
df_clean['Daily_Protein_Intake_g'] = df_clean['Daily_Protein_Intake_g'].fillna(df_clean['Daily_Protein_Intake_g'].median()).round(2)

# ۵. محاسبه شاخص ژنتیکی (Genetic_Score) بین ۰ تا ۱۰۰
scaler = MinMaxScaler(feature_range=(0, 100))
df_clean['Raw_Genetic_Score'] = (df_clean['DXDTOLE'] / df_clean['DXDTOLE'].median()) + \
                                (df_clean['LBDHDD'] / df_clean['LBDHDD'].median()) - \
                                (df_clean['LBXTR'] / df_clean['LBXTR'].median())
df_clean['Genetic_Score'] = scaler.fit_transform(df_clean[['Raw_Genetic_Score']]).round(2)

# ۶. مهندسی سطح فعالیت به دو صورت عددی و متنی (نقشه ۴ سطحی استاندارد)
df_clean['Vigorous_Activity'] = df_clean['PAQ650'].apply(lambda x: 1 if x == 1 else 0)
df_clean['Moderate_Activity'] = df_clean['PAQ665'].apply(lambda x: 1 if x == 1 else 0)
df_clean['Activity_Score'] = df_clean['Vigorous_Activity'] * 2 + df_clean['Moderate_Activity'] + 1 # مقادیر بین ۱ تا ۴

def map_activity_level(score):
    if score == 1: return 'Sedentary'
    elif score == 2: return 'Light'
    elif score == 3: return 'Moderate'
    else: return 'Active'

df_clean['Activity_Level'] = df_clean['Activity_Score'].apply(map_activity_level)

# ۷. محاسبه برچسب نهایی هدف (Protein_Requirement_g) مبتنی بر بافت خالص عضلانی و وزن کل
df_clean['Protein_Requirement_g'] = (df_clean['Lean_Mass_kg'] * 1.2) + (df_clean['BMXWT'] * (df_clean['Activity_Score'] * 0.1)) + (df_clean['Genetic_Score'] / 10)
df_clean['Protein_Requirement_g'] = df_clean['Protein_Requirement_g'].round(2)

# ۸. نگاشت نهایی به جدول ایده‌آل و تغییر نام ستون‌ها
final_mapping = {
    'SEQN': 'ID',
    'RIDAGEYR': 'Age',
    'RIAGENDR': 'Gender',
    'BMXWT': 'Weight_kg',
    'BMXHT': 'Height_cm',
    'BMXBMI': 'BMI',
    'Body_Fat_Percent': 'Body_Fat_Percent',
    'Lean_Mass_kg': 'Lean_Mass_kg',
    'Activity_Level': 'Activity_Level',
    'Activity_Score': 'Activity_Score',
    'Daily_Protein_Intake_g': 'Daily_Protein_Intake_g',
    'Genetic_Score': 'Genetic_Score',
    'Protein_Requirement_g': 'Protein_Requirement_g'
}

df_final_master = df_clean[list(final_mapping.keys())].rename(columns=final_mapping)
df_final_master['Gender'] = df_final_master['Gender'].map({1: 'Male', 2: 'Female'})

# ذخیره در پوشه processed به عنوان مرجع اصلی پروژه
output_master_file = os.path.join("..", "data", "processed", "NHANES_Master_Dataset.csv")
df_final_master.to_csv(output_master_file, index=False)

print("-" * 60)
print(f"✅ دیتاست جامع مرجع با موفقیت ساخته شد!")
print(f"📊 مسیر فایل نهایی: {output_master_file}")
print(f"📐 ابعاد فایل: {df_final_master.shape[0]} سطر و {df_final_master.shape[1]} ستون")
print("-" * 60)
print(df_final_master.head(2).to_string())

در حال ساخت دیتاست مستر با استانداردهای Senior ML Engineer...
------------------------------------------------------------
✅ دیتاست جامع مرجع با موفقیت ساخته شد!
📊 مسیر فایل نهایی: ..\data\processed\NHANES_Master_Dataset.csv
📐 ابعاد فایل: 5856 سطر و 13 ستون
------------------------------------------------------------
        ID   Age  Gender  Weight_kg  Height_cm   BMI  Body_Fat_Percent  Lean_Mass_kg Activity_Level  Activity_Score  Daily_Protein_Intake_g  Genetic_Score  Protein_Requirement_g
2  93705.0  66.0  Female       79.5      158.3  31.7              33.1         49.58          Light               2                   29.26          91.77                  84.57
3  93706.0  18.0    Male       66.3      175.7  21.5              22.7         48.77          Light               2                   94.19          90.89                  80.87


In [8]:
import os
import pandas as pd

# ۱. بارگذاری دیتابیس اصلی از دایرکتوری فعلی
df = pd.read_csv(r'C:\Windows\System32\protein-recommendation-ai\protein-recommendation-ai\data\processed\NHANES_Master_Dataset.csv')

# ۲. اعمال منطق علمی برای ستون نوع مکمل
def assign_supplement(row):
    if row['Activity_Score'] == 1:
        return 'Casein'
    elif row['Body_Fat_Percent'] < 18 and row['Lean_Mass_kg'] > 55:
        return 'Whey_Isolate'
    elif row['BMI'] > 28:
        return 'Plant_Protein'
    else:
        return 'Whey_Concentrate'

df['Recommended_Supplement_Type'] = df.apply(assign_supplement, axis=1)

# ۳. تعریف دقیق مسیر خروجی بر اساس خواسته شما
output_dir = r"C:\Windows\System32\protein-recommendation-ai\protein-recommendation-ai\data\processed"

# ساخت پوشه در صورتی که از قبل وجود نداشته باشد
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

output_path = os.path.join(output_dir, "NHANES_Processed_With_Supplements.csv")

# ۴. ذخیره فایل دیتاسِت جدید
df.to_csv(output_path, index=False)

print(f"🎯 فایل جدید با موفقیت در مسیر زیر ذخیره شد:\n{output_path}")
print("\nتعداد هر مکمل در دیتاسِت:")
print(df['Recommended_Supplement_Type'].value_counts())

🎯 فایل جدید با موفقیت در مسیر زیر ذخیره شد:
C:\Windows\System32\protein-recommendation-ai\protein-recommendation-ai\data\processed\NHANES_Processed_With_Supplements.csv

تعداد هر مکمل در دیتاسِت:
Recommended_Supplement_Type
Casein              3120
Plant_Protein       1397
Whey_Concentrate    1306
Whey_Isolate          33
Name: count, dtype: int64
